# Create dataset using preprocessed data

In [1]:
import csv

with open("processed-actual-outages.csv") as file:
    csv_reader = csv.DictReader(file)
    actual_data = list(csv_reader)

In [ ]:
from datetime import datetime

from tqdm import tqdm

# convert dates to datetime objects, and PTID and Voltage to integers
date_keys = ["OutDatetime", "MinTimeStamp", "MaxTimeStamp"]

for row in tqdm(actual_data):
    row["PTID"] = int(row["PTID"])
    row["Voltage"] = int(row["Voltage"])
    for key in date_keys:
        row[key] = datetime.strptime(row[key], "%Y-%m-%d %H:%M:%S")

actual_data[0]

 94%|█████████▎| 69315/74063 [00:01<00:00, 61603.53it/s]

In [ ]:
from pathlib import Path

output_path = Path("output")
output_path.mkdir(exist_ok=True)

# HYPER PARAMETERS

In [ ]:
EVENT_WINDOW_HOURS = 12
MIN_YEAR = 2008

VOLTAGES = [69, 115, 132, 220, 345, 500, 735]
VOLTAGE_GROUP = {
    # 69 or less
    27: 69,
    34: 69,
    69: 69,
    # 155/120
    115: 115,
    120: 115,
    # 132/138
    132: 132,
    138: 132,
    # 220/230
    220: 220,
    230: 220,
    # 345
    345: 345,
    # 500
    500: 500,
    # 735/765
    735: 735,
    765: 735,
}

INTERVALS_MINUTES = [15, 30, 60, 120]

In [ ]:
actual_data = sorted(
    [
        row
        for row in actual_data
        if row["MinTimeStamp"].year >= MIN_YEAR and row["OutDatetime"].year >= MIN_YEAR
    ],
    key=lambda x: x["MinTimeStamp"],
)

In [ ]:
# load graph
import igraph

g = igraph.Graph.Read_Pickle("res/outage_graph.pkl")

In [ ]:
# load buses/nodes information
import json

bus_name_to_index_path = Path("res/bus_name_to_index.json")
bus_name_to_index = json.loads(bus_name_to_index_path.read_text())

In [ ]:
from datetime import timedelta

import numpy as np


def create_sample(ref_row, window):
    ############
    # features #
    ############
    # simple counting features
    num_events = len(window)
    num_unique_ptids = len(set(row["PTID"] for row in window))
    voltage_group_list = {i: 0 for i in VOLTAGES}
    for row in window:
        voltage_group_list[VOLTAGE_GROUP[row["Voltage"]]] += 1
    voltage_group_list = [voltage_group_list[v] for v in VOLTAGES]
    num_planned = len([row for row in window if row["OutageType"] == "Planned"])
    num_auto = len([row for row in window if row["OutageType"] == "Auto"])
    bus_names = [row[bus] for row in window for bus in ["FirstBus", "SecondBus"]]
    num_unique_buses = len(set(bus_names))

    # fine-grained interval features
    fine_interval_features = []
    for dt in INTERVALS_MINUTES:
        dt = timedelta(minutes=dt)
        num_events_interval = len(
            [
                row
                for row in window
                if ref_row["MinTimeStamp"] - row["MinTimeStamp"] < dt
            ]
        )
        fine_interval_features.append(num_events_interval)

    # graph based features
    node_degrees = np.array(
        [g.degree(bus_name_to_index[bus_name]) for bus_name in bus_names]
    )
    node_degrees_mean = node_degrees.mean().item()
    node_degrees_std = node_degrees.std().item()
    node_degrees_min = node_degrees.min().item()
    node_degrees_max = node_degrees.max().item()
    node_degrees_stats = [
        node_degrees_mean,
        node_degrees_std,
        node_degrees_min,
        node_degrees_max,
    ]

    # aggregate features
    features = [
        num_events,
        num_unique_ptids,
        *voltage_group_list,
        num_planned,
        num_auto,
        num_unique_buses,
        *fine_interval_features,
        *node_degrees_stats,
    ]

    ##########
    # labels #
    ##########
    # Auto/Planned labels
    is_auto = True if ref_row["OutageType"] == "Auto" else False

    # time to reference event
    time_to_event = ref_row["MinTimeStamp"] - max(row["MinTimeStamp"] for row in window)

    # aggregate labels
    labels = [is_auto, time_to_event.total_seconds()]

    return features, labels

In [ ]:
from tqdm import tqdm

# reduce algorithm complexity by only looking at the most recent events in the window
MAX_EVENTS_PER_WINDOW = 100

dataset, ref_rows = [], []

delta_time = timedelta(hours=EVENT_WINDOW_HOURS)
for i, ref_row in enumerate(tqdm(actual_data)):
    # exclude old data
    if (
        ref_row["MinTimeStamp"].year < MIN_YEAR
        or ref_row["OutDatetime"].year < MIN_YEAR
    ):
        continue

    # exclude planned event
    if ref_row["OutageType"] == "Planned":
        continue

    last_scheduled = actual_data[i - 1]
    window = [
        row
        for row in actual_data[max(i - MAX_EVENTS_PER_WINDOW, 0) : i]
        if last_scheduled["MinTimeStamp"] - row["MinTimeStamp"] < delta_time
    ]
    if window:
        dataset.append(create_sample(ref_row, window))
        ref_rows.append(ref_row)

len(dataset)

In [ ]:
features, labels = dataset[0]
features, labels

In [ ]:
from sklearn.model_selection import train_test_split, KFold


def assign_fold_ids(dataset, test_size=0.20, n_splits=5, seed=42):
    """
    Assign fold_id to each sample in a dataset.

    fold_id = -1  -> held-out test set
    fold_id = 0-4 -> 5-fold CV folds inside the remaining train/validation set

    Parameters
    ----------
    dataset : list-like
        Dataset where each item is like:
            features, label = dataset[i]

    test_size : float
        Proportion of data reserved as held-out test set.

    n_splits : int
        Number of CV folds.

    seed : int
        Random seed for reproducibility.

    Returns
    -------
    fold_ids : np.ndarray
        Array of shape (len(dataset),), containing fold IDs.
    """

    n_samples = len(dataset)
    indices = np.arange(n_samples)

    # Initialise all samples as unassigned
    fold_ids = np.empty(n_samples, dtype=int)

    # Step 1: create held-out test set
    trainval_idx, test_idx = train_test_split(
        indices,
        test_size=test_size,
        random_state=seed,
        shuffle=True,
    )

    # Assign held-out test samples
    fold_ids[test_idx] = -1

    # Step 2: assign 5-fold IDs inside train/validation set
    kfold = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )

    for fold_id, (_, val_idx_relative) in enumerate(kfold.split(trainval_idx)):
        val_idx_absolute = trainval_idx[val_idx_relative]
        fold_ids[val_idx_absolute] = fold_id

    return fold_ids

# Provide "where" label

In [ ]:
import random
import numpy as np
import pandas as pd
import igraph as ig


def assign_line_zones(
    g: ig.Graph,
    seed: int = 42,
    objective_function: str = "modularity",
    inter_zone_policy: str = "boundary_pair",
):
    """
    Assign reproducible community-based zones to vertices and lines/edges.

    Parameters
    ----------
    g : igraph.Graph
        Power network graph. Vertices are buses/substations and edges are lines.

    seed : int
        Random seed for reproducibility.

    objective_function : str
        Leiden objective function. Usually "modularity" or "CPM".

    inter_zone_policy : {"boundary", "boundary_pair", "lower_zone"}
        How to assign an edge whose endpoints belong to different communities.

        "boundary":
            all inter-community edges get zone_id = -1

        "boundary_pair":
            inter-community edge gets zone label like "2--7"

        "lower_zone":
            inter-community edge is assigned to min(zone_u, zone_v)

    Returns
    -------
    communities : igraph.VertexClustering
        Leiden community object.

    edge_zone_df : pandas.DataFrame
        DataFrame containing edge ID, endpoints, endpoint communities, and line zone.

    g : igraph.Graph
        Same graph with vertex and edge attributes added:
            vertex attribute: "zone_id"
            edge attributes: "zone_id", "is_inter_zone"
    """

    if inter_zone_policy not in {"boundary", "boundary_pair", "lower_zone"}:
        raise ValueError(
            "inter_zone_policy must be one of: "
            "'boundary', 'boundary_pair', or 'lower_zone'"
        )

    # Reproducibility for igraph Leiden
    random.seed(seed)
    np.random.seed(seed)

    communities = g.community_leiden(objective_function=objective_function)

    raw_membership = list(communities.membership)
    n_communities = len(set(raw_membership))

    # Renumber communities by decreasing size.
    # This makes zone labels more stable and interpretable:
    # zone 0 = largest community, zone 1 = second largest, etc.
    community_sizes = pd.Series(raw_membership).value_counts()
    old_to_new = {
        old_cid: new_cid
        for new_cid, old_cid in enumerate(community_sizes.index.tolist())
    }

    vertex_zone = [old_to_new[cid] for cid in raw_membership]
    g.vs["zone_id"] = vertex_zone

    edge_records = []

    for eid, edge in enumerate(g.es):
        u, v = edge.tuple

        zone_u = vertex_zone[u]
        zone_v = vertex_zone[v]

        is_inter_zone = zone_u != zone_v

        if not is_inter_zone:
            line_zone = zone_u
        else:
            if inter_zone_policy == "boundary":
                line_zone = -1
            elif inter_zone_policy == "boundary_pair":
                line_zone = f"{min(zone_u, zone_v)}--{max(zone_u, zone_v)}"
            elif inter_zone_policy == "lower_zone":
                line_zone = min(zone_u, zone_v)

        edge["zone_id"] = line_zone
        edge["is_inter_zone"] = is_inter_zone

        edge_records.append(
            {
                "edge_id": eid,
                "from_vertex": u,
                "to_vertex": v,
                "from_name": g.vs[u]["name"] if "name" in g.vs.attributes() else u,
                "to_name": g.vs[v]["name"] if "name" in g.vs.attributes() else v,
                "from_zone": zone_u,
                "to_zone": zone_v,
                "line_zone": line_zone,
                "is_inter_zone": is_inter_zone,
            }
        )

    edge_zone_df = pd.DataFrame(edge_records)

    return communities, edge_zone_df

In [ ]:
communities, edge_zone_df = assign_line_zones(
    g,
    seed=42,
    objective_function="modularity",
    inter_zone_policy="boundary_pair",
)

edge_zone_df.to_csv(output_path / "edge_zones.csv", index=False)

In [ ]:
# import matplotlib.pyplot as plt
# from matplotlib.patches import Patch

# community_ids = communities.membership
# num_communities = len(communities)

# cmap = plt.colormaps["gist_ncar"]
# denominator = max(num_communities - 1, 1)
# vertex_colors = [cmap(community_id / denominator) for community_id in community_ids]

# layout = g.layout_kamada_kawai()

# fig, ax = plt.subplots(figsize=(16, 16))
# igraph.plot(
#     g,
#     target=ax,
#     layout=layout,
#     vertex_color=vertex_colors,
#     vertex_size=5,
#     vertex_frame_width=0,
#     edge_color=(0.6, 0.6, 0.6, 0.25),
#     edge_width=0.4,
# )
# ax.set_title(f"NYISO topology communities ({num_communities} communities, modularity={communities.modularity:.3f})")
# ax.set_axis_off()

# community_sizes = communities.sizes()
# largest_communities = sorted(range(num_communities), key=lambda cid: community_sizes[cid], reverse=True)[:10]
# legend_handles = [
#     Patch(color=cmap(cid / denominator), label=f"Community {cid} ({community_sizes[cid]} nodes)")
#     for cid in largest_communities
# ]
# ax.legend(handles=legend_handles, loc="upper right", frameon=False, title="Largest communities")

# community_graph_path = output_path / "nyiso_graph_communities.png"
# fig.savefig(community_graph_path, dpi=300, bbox_inches="tight")
# plt.show()

# community_graph_path

In [ ]:
LABEL_INDEX = 1

for i, ref_row in enumerate(ref_rows):
    first_bus = ref_row["FirstBus"]
    second_bus = ref_row["SecondBus"]
    from_zone = g.vs[bus_name_to_index[first_bus]]["zone_id"]
    to_zone = g.vs[bus_name_to_index[second_bus]]["zone_id"]
    dataset[i][LABEL_INDEX].extend([from_zone, to_zone])

In [ ]:
dataset[0]

# save the dataset

In [ ]:
from collections import Counter

feature_columns = [
    "num_events",
    "num_unique_ptids",
    *[f"num_{voltage}kv_lines" for voltage in VOLTAGES],
    "num_planned_outages",
    "num_auto_outages",
    "num_unique_buses",
    *[f"num_events_last_{minutes}_min" for minutes in INTERVALS_MINUTES],
    "node_degree_mean",
    "node_degree_std",
    "node_degree_min",
    "node_degree_max",
]
label_columns = [
    "label_is_auto",
    "label_time_to_event_seconds",
    "label_from_zone",
    "label_to_zone",
]
columns = ["fold_id", *feature_columns, *label_columns]

fold_ids = assign_fold_ids(dataset)
dataset_csv_path = output_path / f"dataset_winsize{EVENT_WINDOW_HOURS:02d}h.csv"

with dataset_csv_path.open("w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(columns)
    for fold_id, (features, labels) in zip(fold_ids, dataset):
        if len(features) != len(feature_columns) or len(labels) != len(label_columns):
            raise ValueError(
                "Feature or label column count does not match the dataset sample shape."
            )
        writer.writerow([fold_id, *features, *labels])

dataset_csv_path, Counter(fold_ids)